# ⚽ FWC 26 — Traffic Analytics Dashboard

---

## 📖 Guide rapide — Pour les équipes métiers (sans toucher au code !)

### Étape 1 — Ouvrir ce notebook
Clique sur le lien partagé. Google Colab s'ouvre dans ton navigateur, **aucune installation requise**.

### Étape 2 — Exécuter tout le code
Dans le menu en haut, clique sur **`Exécution` → `Tout exécuter`** (ou `Runtime → Run all`).

> Une fenêtre peut te demander de confirmer ("Ce notebook n'a pas été écrit par Google…") → clique **Exécuter quand même**.

### Étape 3 — Lire le graphique
- **Courbe bleue** → nombre de visiteurs (axe gauche)
- **Courbe orange pointillée** → durée moyenne de session en minutes (axe droit)
- **Lignes rouges verticales** → matchs FWC 26 (le nom s'affiche en haut)
- **Zoomer** : dessine un rectangle avec la souris sur n'importe quel pic
- **Dézoomer** : double-clic sur le graphique, ou boutons en haut à droite

### Étape 4 — Mettre à jour
Pour recharger les dernières données du tableur → relance **`Tout exécuter`**.

---

In [ ]:
# ── 0. Installation des bibliothèques (exécuté une seule fois) ────────────────
# Plotly est déjà disponible dans Colab, on l'importe directement.
# Si une erreur "ModuleNotFoundError" apparaît, décommente la ligne pip install.

# !pip install -q plotly pandas

In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import re
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

In [ ]:
# ── 2. Configuration — seule cellule à modifier si nécessaire ─────────────────

SHEET_ID   = "1j3PNyXzN28Ll3L8zW7Rx5UAXvCQz1yMHQ3Jcz_Z1B7U"
SHEET_NAME = "FWC 26 - Chart data"   # nom exact de l'onglet
YEAR       = 2026                     # année courante pour reconstruire les dates

# URL de téléchargement CSV public (aucun identifiant requis)
CSV_URL = (
    f"https://docs.google.com/spreadsheets/d/{SHEET_ID}"
    f"/gviz/tq?tqx=out:csv&sheet={SHEET_NAME.replace(' ', '%20')}"
)
print("✅ Configuration OK")

In [ ]:
# ── 3. Chargement brut depuis Google Sheets ───────────────────────────────────

print("📡 Chargement des données…")
raw = pd.read_csv(CSV_URL, header=0)
raw.columns = [c.strip() for c in raw.columns]

print(f"✅ {len(raw)} lignes chargées — colonnes : {list(raw.columns)}")
raw.head(10)

In [ ]:
# ── 4. Nettoyage avancé — Forward Fill des dates ──────────────────────────────
#
# Problème : la colonne Date ressemble à :
#   "12 Jun 00:00"   ← première heure du jour, contient la date complète
#   "01:00"          ← heure seule, la date doit être propagée depuis la ligne du dessus
#   "02:00"
#   ...
#   "13 Jun 00:00"   ← nouveau jour
#
# Solution : on détecte le pattern complet, on mémorise le "jour courant",
# puis on le préfixe sur toutes les lignes horaires suivantes.

MONTH_MAP = {
    "jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6,
    "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12,
}

FULL_DATE_RE = re.compile(r"^\s*(\d{1,2})\s+([A-Za-z]{3})\s+(\d{1,2}:\d{2})\s*$")
TIME_ONLY_RE = re.compile(r"^\s*(\d{1,2}:\d{2})\s*$")


def forward_fill_dates(series: pd.Series) -> pd.Series:
    """Reconstruct full datetime strings using forward-fill logic."""
    current_day = None
    result = []
    for raw_val in series:
        s = str(raw_val).strip() if not pd.isna(raw_val) else ""
        m_full = FULL_DATE_RE.match(s)
        m_time = TIME_ONLY_RE.match(s)
        if m_full:
            day   = m_full.group(1).zfill(2)
            month = m_full.group(2).capitalize()
            time  = m_full.group(3)
            current_day = f"{day} {month}"
            result.append(f"{current_day} {time}")
        elif m_time and current_day:
            result.append(f"{current_day} {m_time.group(1)}")
        else:
            result.append(None)
    return pd.Series(result, index=series.index)


def parse_datetime(s):
    """Parse '12 Jun 00:00' → pd.Timestamp."""
    if s is None or pd.isna(s):
        return pd.NaT
    try:
        parts = str(s).strip().split()
        day   = int(parts[0])
        month = MONTH_MAP.get(parts[1].lower(), 0)
        hh, mm = map(int, parts[2].split(":"))
        return pd.Timestamp(YEAR, month, day, hh, mm)
    except Exception:
        return pd.NaT


def duration_to_minutes(value) -> float | None:
    """Convert 'HH:MM:SS' or 'MM:SS' to decimal minutes."""
    if pd.isna(value) or str(value).strip() == "":
        return None
    parts = str(value).strip().split(":")
    try:
        if len(parts) == 3:
            h, m, s = int(parts[0]), int(parts[1]), float(parts[2])
            return h * 60 + m + s / 60
        if len(parts) == 2:
            m, s = int(parts[0]), float(parts[1])
            return m + s / 60
    except ValueError:
        pass
    return None


# ── Appliquer les transformations ─────────────────────────────────────────────
df = raw.copy()

df["_date_str"] = forward_fill_dates(df["Date"])
df["DateTime"]  = df["_date_str"].apply(parse_datetime)

df["Total Visitors"] = (
    df["Total Visitors"].astype(str)
    .str.replace(",", "", regex=False).str.strip()
)
df["Total Visitors"] = pd.to_numeric(df["Total Visitors"], errors="coerce")

df["Session (min)"] = df["Average session time"].apply(duration_to_minutes)

# Colonne match (FWC ou BBC)
match_col = next((c for c in ["FWC Matches", "BBC Matches"] if c in df.columns), None)
df["Match"] = df[match_col].astype(str).str.strip().replace({"nan": "", "None": ""}) if match_col else ""

df = df.dropna(subset=["DateTime"]).sort_values("DateTime").reset_index(drop=True)

print(f"✅ Données nettoyées : {len(df)} lignes valides")
print(f"   Période : {df['DateTime'].min()} → {df['DateTime'].max()}")
print(f"   Matchs détectés : {df[df['Match'] != '']['Match'].nunique()}")
df[["DateTime", "Total Visitors", "Session (min)", "Match"]].head(10)

In [ ]:
# ── 5. Graphique interactif double axe ────────────────────────────────────────

fig = go.Figure()

# ── Courbe 1 : Visiteurs (axe Y gauche) ───────────────────────────────────────
fig.add_trace(go.Scatter(
    x=df["DateTime"],
    y=df["Total Visitors"],
    name="Total Visitors",
    yaxis="y1",
    mode="lines",
    line=dict(color="#1a6cdb", width=2),
    fill="tozeroy",
    fillcolor="rgba(26,108,219,0.12)",
    hovertemplate="<b>%{x|%d %b %H:%M}</b><br>Visiteurs : <b>%{y:,}</b><extra></extra>",
))

# ── Courbe 2 : Session (axe Y droit) ─────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=df["DateTime"],
    y=df["Session (min)"],
    name="Session moy. (min)",
    yaxis="y2",
    mode="lines",
    line=dict(color="#e88a00", width=2, dash="dot"),
    hovertemplate="<b>%{x|%d %b %H:%M}</b><br>Session : <b>%{y:.1f} min</b><extra></extra>",
))

# ── Lignes verticales + annotations pour chaque match ────────────────────────
shapes, annotations = [], []
matches_df = df[df["Match"] != ""].copy()

for _, row in matches_df.iterrows():
    x_val = row["DateTime"]
    label = row["Match"]
    shapes.append(dict(
        type="line", x0=x_val, x1=x_val,
        yref="paper", y0=0, y1=1,
        line=dict(color="rgba(200,30,30,0.6)", width=1.5, dash="dash"),
    ))
    annotations.append(dict(
        x=x_val, yref="paper", y=1.01,
        text=f"⚽ {label}",
        showarrow=False,
        textangle=-45,
        font=dict(size=8.5, color="#c01e1e"),
        xanchor="left",
    ))

# ── Boutons de zoom rapide ────────────────────────────────────────────────────
range_buttons = [
    dict(count=1,  label="24 h",    step="day",  stepmode="backward"),
    dict(count=3,  label="3 jours", step="day",  stepmode="backward"),
    dict(count=7,  label="7 jours", step="day",  stepmode="backward"),
    dict(step="all", label="Vue globale"),
]

fig.update_layout(
    title=dict(
        text="<b>FWC 26 — Trafic horaire</b>   <sup>Visiteurs & Durée de session</sup>",
        font=dict(size=20),
    ),
    height=600,
    hovermode="x unified",
    shapes=shapes,
    annotations=annotations,
    legend=dict(orientation="h", y=1.06, x=0),
    plot_bgcolor="#0e1117",
    paper_bgcolor="#0e1117",
    font=dict(color="#fafafa"),
    xaxis=dict(
        title="Date / Heure",
        gridcolor="#2a2f3d",
        rangeselector=dict(
            buttons=range_buttons,
            bgcolor="#1a1f2e",
            activecolor="#c01e1e",
            font=dict(color="#fafafa"),
        ),
        rangeslider=dict(visible=True, thickness=0.06, bgcolor="#1a1f2e"),
        type="date",
    ),
    yaxis=dict(
        title="Total Visitors",
        title_font=dict(color="#1a6cdb"),
        tickfont=dict(color="#1a6cdb"),
        gridcolor="#2a2f3d",
        rangemode="tozero",
    ),
    yaxis2=dict(
        title="Session moyenne (min)",
        title_font=dict(color="#e88a00"),
        tickfont=dict(color="#e88a00"),
        overlaying="y",
        side="right",
        rangemode="tozero",
        showgrid=False,
    ),
    margin=dict(t=110, r=70, b=50, l=70),
    dragmode="zoom",   # rectangle zoom par défaut
)

fig.show()

In [ ]:
# ── 6. Tableau des matchs détectés ───────────────────────────────────────────
# Pratique pour vérifier rapidement quels matchs ont été identifiés.

match_summary = (
    df[df["Match"] != ""][["DateTime", "Match", "Total Visitors", "Session (min)"]]
    .rename(columns={"DateTime": "Heure du match"})
    .reset_index(drop=True)
)
print(f"📋 {len(match_summary)} événement(s) match trouvé(s) :")
display(match_summary)

In [ ]:
# ── 7. Statistiques rapides ───────────────────────────────────────────────────

print("=" * 50)
print("📊 RÉSUMÉ STATISTIQUE")
print("=" * 50)
print(f"Période couverte  : {df['DateTime'].min().strftime('%d %b %Y %H:%M')} "
      f"→ {df['DateTime'].max().strftime('%d %b %Y %H:%M')}")
print(f"Total visiteurs   : {df['Total Visitors'].sum():,.0f}")
print(f"Pic de trafic     : {df['Total Visitors'].max():,} "
      f"({df.loc[df['Total Visitors'].idxmax(), 'DateTime'].strftime('%d %b %H:%M')})")
print(f"Session moy. glob.: {df['Session (min)'].mean():.1f} min")
print(f"Matchs détectés   : {df[df['Match'] != '']['Match'].nunique()}")